# SI7011 · Sesión 06 · Generar las respuestas con estilo del integrador (solo profesor)

Produce `dolly_quijote_es.csv`, el dataset de Kaggle `juanmartinezv4399/si7011-s06-dolly-quijote` que lee el integrador 6.6: unas 1 000 preguntas de Dolly-15k en español (`argilla/databricks-dolly-15k-curated-multilingual`, split `es`), **distintas** de las 2 200 del notebook 6.3 (mismo filtro y mismo orden, posiciones 2 200 a 3 699), con su respuesta reescrita en el castellano de *Don Quijote* por Qwen2.5-7B-Instruct cuantizado a 4 bits. Se generan 1 250 + 250 y se descartan las respuestas con menos de dos giros de la época o con largo raro; el integrador separa 100 de `train` para validar.

**Cómo correrlo en Kaggle**
1. *Settings*: GPU T4 e **Internet** activado.
2. **Save Version → Save & Run All (Commit)**. Tarda unos 40 minutos (estimado).
3. Revisa las muestras del final. Descarga `dolly_quijote_es.csv` de *Output* y publícalo como dataset `si7011-s06-dolly-quijote`, público, licencia CC BY-SA 3.0 (la de Dolly).

In [ ]:
# Kaggle trae torchao 0.10, que las versiones recientes de transformers y peft rechazan al importarse
# ("Found an incompatible version of torchao"). Este notebook no lo usa: si es viejo, se desinstala.
import os
from importlib.metadata import version, PackageNotFoundError
try:
    if tuple(int(x) for x in version('torchao').split('.')[:2]) < (0, 16):
        os.system('pip uninstall -y -q torchao')
except (PackageNotFoundError, ValueError):
    pass

import json, os, re, time
import pandas as pd
import torch
try:
    import bitsandbytes  # noqa: F401
except ImportError:
    os.system('pip install -q bitsandbytes')
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import load_dataset
dev = 'cuda'
assert torch.cuda.is_available(), 'Necesita GPU (Kaggle T4).'

In [ ]:
DOLLY = 'argilla/databricks-dolly-15k-curated-multilingual'
N_TRAIN, N_TEST = 1250, 250     # el filtro de estilo descarta parte; quedan unas 1 000 + 200
INICIO = 2200        # el notebook 6.3 usa las posiciones 0 a 2199 de este mismo orden: aquí no se repite ninguna

def util(e):         # el mismo filtro del notebook 6.3
    return (not (e['context'] or '').strip() and 20 <= len(e['response']) <= 700
            and len(e['instruction']) <= 300)

filtrado = load_dataset(DOLLY, split='es').filter(util).shuffle(seed=0)
sel = filtrado.select(range(INICIO, INICIO + N_TRAIN + N_TEST))
print(len(filtrado), 'útiles ·', len(sel), 'elegidas')

In [ ]:
GEN = 'Qwen/Qwen2.5-7B-Instruct'
tok = AutoTokenizer.from_pretrained(GEN, padding_side='left')
gen = AutoModelForCausalLM.from_pretrained(
    GEN, device_map='auto',
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                                           bnb_4bit_quant_type='nf4')).eval()

In [ ]:
SISTEMA = ('Reescribes respuestas en el castellano de Don Quijote de la Mancha (principios del siglo XVII). '
           'Reglas: conserva exactamente el contenido y los datos de la respuesta original, sin agregar hechos; '
           'empieza SIEMPRE dirigiéndote al lector ("Sabed, vuestra merced, que...", "Os digo, señor mío, que...", '
           '"En verdad os digo que...") y usa en cada respuesta al menos tres giros de la época (vuestra merced, '
           'agora, mesmo, asaz, aqueste, hube de, os digo, en verdad, pardiez, sabed); tono cortés y algo '
           'grandilocuente; largo parecido al original; no menciones a don Quijote, a Sancho ni a Cervantes; '
           'escribe solo la respuesta reescrita.')
EJEMPLOS = [
    ('¿Qué es un átomo?',
     'Un átomo es la unidad más pequeña de un elemento químico que conserva sus propiedades.',
     'Sabed, vuestra merced, que el átomo es la más menuda parte de un elemento que, aun así de pequeña, '
     'guarda en sí todas las propiedades de aquel, como guarda el grano la virtud de la espiga.'),
    ('Dame tres ideas para ahorrar agua en casa.',
     'Cierra la llave mientras te cepillas los dientes, repara las fugas y usa la lavadora solo con carga completa.',
     'Tres consejos os doy, y no de poca monta: cerrad la llave en tanto os limpiáis los dientes; remediad '
     'luego cualquier gotera o fuga que en la casa hubiere; y no echéis a andar la lavadora sino cuando '
     'vaya bien colmada de ropa.'),
]

def mensajes(pregunta, respuesta):
    m = [{'role': 'system', 'content': SISTEMA}]
    for p, r, e in EJEMPLOS:
        m += [{'role': 'user', 'content': f'Pregunta: {p}\nRespuesta original: {r}'}, {'role': 'assistant', 'content': e}]
    return m + [{'role': 'user', 'content': f'Pregunta: {pregunta}\nRespuesta original: {respuesta}'}]

In [ ]:
@torch.no_grad()
def reescribir(lote, max_nuevos=320):
    textos = [tok.apply_chat_template(mensajes(e['instruction'], e['response']), tokenize=False,
                                      add_generation_prompt=True) for e in lote]
    x = tok(textos, return_tensors='pt', padding=True).to(gen.device)
    out = gen.generate(**x, max_new_tokens=max_nuevos, do_sample=False, repetition_penalty=1.05,
                       pad_token_id=tok.eos_token_id)
    return [tok.decode(o[x['input_ids'].shape[1]:], skip_special_tokens=True).strip() for o in out]

B, filas, t0 = 16, [], time.time()
for i in range(0, len(sel), B):
    lote = [sel[j] for j in range(i, min(i + B, len(sel)))]
    for e, r in zip(lote, reescribir(lote)):
        filas.append({'id': e['id'], 'category': e['category'], 'instruction': e['instruction'],
                      'response': e['response'], 'response_estilo': r,
                      'split': 'train' if len(filas) < N_TRAIN else 'test'})
    if i % (10 * B) == 0:
        print(f'{len(filas)}/{len(sel)} · {time.time() - t0:.0f} s · {filas[-1]["response_estilo"][:120]}')

In [ ]:
df = pd.DataFrame(filas)
razon = df['response_estilo'].str.len() / df['response'].str.len()
ok = (df['response_estilo'].str.len() > 10) & razon.between(0.5, 2.5) & \
     ~df['response_estilo'].str.contains(r'Respuesta original|Pregunta:|Reescrit', regex=True)
GIROS = ['vuestra merced', 'agora', 'mesmo', 'asaz', 'os digo', 'pardiez', 'aqueste', 'hube de', 'sabed', 'en verdad']          # los mismos del juez del integrador
n_giros = df['response_estilo'].str.lower().apply(lambda r: sum(g in r for g in GIROS))
print('giros por respuesta:', n_giros.value_counts().sort_index().to_dict())
print(f'{(~ok).sum()} filas con largo raro o texto del prompt · {(ok & (n_giros < 2)).sum()} más con menos de 2 giros')
ok &= n_giros >= 2                                         # el estilo tiene que notarse
df = df[ok].reset_index(drop=True)
df.to_csv('dolly_quijote_es.csv', index=False)
print(df['split'].value_counts().to_dict())
for _, r in df.sample(4, random_state=0).iterrows():
    print('\n▶', r['instruction'], '\n  original:', r['response'][:200], '\n  estilo:  ', r['response_estilo'][:300])